# 22DM015 Final Project — Financial PhraseBank

**Data Split + Part 2 augmentation/generation**

This notebook downloads the data and divides it into train, val, and test sets that will be used by all models.‍ It also prepares the 32-labelled dataset and the augmented datasets (with and without an LLM) required by Part 2.‍

To prepare the data, we created the Python module `data_utils`, which sets global configuration used by all notebooks such as `SEED` and `LABEL_NAMES`.‍ The module also contains helper functions to download the data, split it, and then load the split data.‍

# A.‍ Preparations

In [1]:
# watermark: AGLLM (AI-assisted content disclosure)
import sys, os

# Add the parent directory to the sys.path to import data_utils
sys.path.append(os.path.abspath('..'))
import data_utils as du

# Global config, call from data_utils so consistent across notebooks
SEED = du.SEED
print(f"SEED: {SEED}")

DATA_DIR = du.DATA_DIR
print(f"DATA_DIR: {DATA_DIR}")

SEED: 618
DATA_DIR: /Users/cherrylchico/Desktop/BSE/T3/Advanced Methods in NLP/advanced_nlp/data
Last run: 2026-06-16 19:01:12


In [3]:
# watermark: AGLLM (AI-assisted content disclosure)
import re, html, json, random, time
from pathlib import Path

import pandas as pd

import nltk
from nltk.corpus import wordnet as wn

import torch, transformers
from transformers import MarianMTModel, MarianTokenizer

from groq import Groq

# personal preference set-up: show multiple cell outputs
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

Last run: 2026-06-16 19:01:31


# B.‍ Build the Data Split

We call the function `build_splits` from the `data_utils` module.‍ The function downloads the `Financial_phrasebank` dataset (`Sentences_AllAgree` version) from Hugging Face and then splits it into **train**, **validation**, and **test** sets.‍ Behind the scenes, it uses `hf_hub_download` from the `huggingface_hub` module to directly download `takala/financial_phrasebank`, and then uses `train_test_split` from `sklearn.model_selection` to divide the downloaded data according to the user-supplied test and validation percentages.‍ The divided data is then saved in the `data` folder.‍

We chose a 70/10/20 train/validation/test split: 20% of the data is held out as test, 10% of the whole dataset as validation, and the remaining 70% is used for training.‍ The split is stratified per sentiment so we should have consistent distribution across splits.‍ 

In [3]:
# watermark: AGLLM (AI-assisted content disclosure)
splits = du.build_splits(
    out_dir=du.DATA_DIR, 
    test_size=0.2, 
    val_size=0.1,
    seed=SEED
)                          

Last run: 2026-06-15 13:01:49


In [4]:
# watermark: AGLLM (AI-assisted content disclosure)
#Another helper function to load the splits
splits = du.load_splits(du.DATA_DIR)
splits.keys()

dict_keys(['train', 'val', 'test'])

Last run: 2026-06-16 19:01:43


In [5]:
# watermark: AGLLM (AI-assisted content disclosure)
for key, df in splits.items():
    print(f"{key}: {df.shape}")
    print(df.head())
    print("-----")

train: (1584, 4)
   id                                               text  label label_name
0   0  According to Gran , the company has no plans t...      1    neutral
1   1  For the last quarter of 2010 , Componenta 's n...      2   positive
2   2  In the third quarter of 2010 , net sales incre...      2   positive
3   5  Finnish Talentum reports its operating profit ...      2   positive
4   6  Clothing retail chain Sepp+ñl+ñ 's sales incre...      2   positive
-----
val: (227, 4)
   id                                               text  label label_name
0   8  Foundries division reports its sales increased...      2   positive
1  11  MegaFon 's subscriber base increased 16.1 % in...      2   positive
2  21  The fair value of the property portfolio doubl...      2   positive
3  31  The company 's order book stood at 1.5 bln eur...      2   positive
4  35  Shares of Nokia Corp. rose Thursday after the ...      2   positive
-----
test: (453, 4)
   id                                     

In [6]:
# watermark: AGLLM (AI-assisted content disclosure)
#Counts
stats_df = pd.concat([df.assign(split=key) for key, df in splits.items()]).pivot_table(index="split", columns="label_name", values="id", aggfunc="count", margins=True, margins_name="total")
stats_df

label_name,negative,neutral,positive,total
split,,,,
test,61,278,114,453
train,212,973,399,1584
val,30,140,57,227
total,303,1391,570,2264


Last run: 2026-06-16 19:01:45


In [7]:
# watermark: AGLLM (AI-assisted content disclosure)
# Confirming distribution is as supplied
(stats_df[['total']] / stats_df.loc["total", "total"] * 100).round(2) 

label_name,total
split,
test,20.01
train,69.96
val,10.03
total,100.00


Last run: 2026-06-16 19:01:46


In [8]:
# watermark: AGLLM (AI-assisted content disclosure)
# Distribution of sentiment per split is consistent with overall distribution
(stats_df.div(stats_df["total"], axis=0) * 100).round(2)


label_name,negative,neutral,positive,total
split,,,,
test,13.47,61.37,25.17,100.0
train,13.38,61.43,25.19,100.0
val,13.22,61.67,25.11,100.0
total,13.38,61.44,25.18,100.0


Last run: 2026-06-16 19:01:47


# C.‍ Sample Labeled 32 

We use another helper function from `data_utils` called `sample_few_shot`, which samples `k` records from a given dataset according to a supplied distribution of sentiment.‍

For this analysis, as the data is imbalanced (~61% neutral, ~25% positive, and ~13% negative), we choose to create a balanced sample to maximize the chance per sentiment given the small sample.‍ For a 32-sample, the distribution is

    NSHOT_PER_CLASS = {"negative": 11, "neutral": 10, "positive": 11}

We sample the 32-labeled set from the train dataset.‍

In [9]:
# watermark: AGLLM (AI-assisted content disclosure)
NSHOT_PER_CLASS = {"negative": 11, "neutral": 10, "positive": 11}

labeled_32 = du.sample_few_shot(
    splits['train'],
    n_per_class=NSHOT_PER_CLASS
)    
labeled_32.to_csv('../data/labeled_32.csv', index=False)

Last run: 2026-06-16 19:01:51


In [10]:
# watermark: AGLLM (AI-assisted content disclosure)
labeled_32.groupby('label_name').size()

label_name
negative    11
neutral     10
positive    11
dtype: int64

Last run: 2026-06-16 19:01:51


# Part 2b.‍ Dataset Augmentation WITHOUT an LLM (1)

Our goal is to augment the 32 financial texts, meaning we want to create more examples out of the ones we have without changing the sentiment each text expresses.‍ Our criterion for augmentation is that the generated data should be diverse in terms of words (i.e.‍ add new tokens that carry sentiment) and that shuffling of words is acceptable (because sentences are treated as series of tokens).‍

We are aware of specific nuances in using financial phrases, and we will work through them by observing some examples.‍

In [11]:
# watermark: AGLLM (AI-assisted content disclosure)
# Ensure the wordnet corpus is available (used by the synonym checks below)
try:
    wn.synsets('test')
except LookupError:
    nltk.download('wordnet'); nltk.download('omw-1.4')

DEVICE = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')

[Synset('trial.n.02'),
 Synset('test.n.02'),
 Synset('examination.n.02'),
 Synset('test.n.04'),
 Synset('test.n.05'),
 Synset('test.n.06'),
 Synset('test.v.01'),
 Synset('screen.v.01'),
 Synset('quiz.v.01'),
 Synset('test.v.04'),
 Synset('test.v.05'),
 Synset('test.v.06'),
 Synset('test.v.07')]

Last run: 2026-06-16 19:01:57


In [12]:
# watermark: AGLLM (AI-assisted content disclosure)
df = pd.read_csv("../data/labeled_32.csv")
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   id          32 non-null     int64
 1   text        32 non-null     str  
 2   label       32 non-null     int64
 3   label_name  32 non-null     str  
dtypes: int64(2), str(2)
memory usage: 5.2 KB


,id,text,label,label_name
0,6,Clothing retail chain Sepp+ñl+ñ 's sales incre...,2,positive
1,26,"In January-September 2010 , Fiskars ' net prof...",2,positive
2,27,Net income from life insurance rose to EUR 16....,2,positive
3,99,Both operating profit and sales for the three-...,2,positive
4,117,Finnish department store chain Stockmann Oyj A...,2,positive


Last run: 2026-06-16 19:01:58


In [13]:
# watermark: AGLLM (AI-assisted content disclosure)
# Print first 5 examples of each label
labels = df['label_name'].unique()
for label in labels:
    print(f"\nLabel: {label}\n")
    examples = df[df['label_name'] == label]['text'].tolist()
    for i, example in enumerate(examples[:5]):  
        print(f"- {example}\n")


Label: positive

- Clothing retail chain Sepp+ñl+ñ 's sales increased by 8 % to EUR 155.2 mn , and operating profit rose to EUR 31.1 mn from EUR 17.1 mn in 2004 .

- In January-September 2010 , Fiskars ' net profit went up by 14 % year-on-year to EUR 65.4 million and net sales to EUR 525.3 million from EUR 487.7 million .

- Net income from life insurance rose to EUR 16.5 mn from EUR 14.0 mn , and net income from non-life insurance to EUR 22.6 mn from EUR 15.2 mn in 2009 .

- Both operating profit and sales for the three-month period increased , respectively from EUR0 .3 m and EUR13 .1 m , as compared to the corresponding period in 2005 .

- Finnish department store chain Stockmann Oyj Abp net profit rose to 39.8 mln euro ( $ 56.8 mln ) for the first nine months of 2007 from 37.4 mln euro ( $ 53.4 mln ) for the same period of 2006 .


Label: neutral

- The company generates net sales of about 600 mln euro $ 775.5 mln annually and employs 6,000 .

- You need to be ready when the window

## Exploratory Data Analysis

### Obs 1: Finance words that have non-financial synonyms
We list some candidate words that are specific to finance but have a different meaning outside finance.‍ For each of the words, we check their existence in the 32, and then generate the synonyms based on the `wordnet` corpus from nltk.‍

We found that the words `share`, `margin`, and `stock` are in the 32-labelled dataset, and that these words contain a lot of synonyms unrelated to their meaning in finance.‍


**Action:** We cannot simply do a word replacement via synonyms.‍

In [14]:
# watermark: AGLLM (AI-assisted content disclosure)
candidates = ['interest', 'share', 'shares', 'bond', 'security', 'margin',
              'return', 'principal', 'capital', 'stock', 'note']

Last run: 2026-06-16 19:02:02


In [15]:
# watermark: AGLLM (AI-assisted content disclosure)
rows = []
for w in candidates:
    in_32 = bool(df['text'].str.contains(rf'\b{w}\b', case=False, regex=True).any())
    syns = []
    for s in wn.synsets(w):
        for lem in s.lemmas():
            name = lem.name().replace('_', ' ')
            if name.lower() != w and name not in syns:
                syns.append(name)
    rows.append({'term': w, 'in_32_texts': in_32,
                 'n_wordnet_senses': len(wn.synsets(w)), 'sample_synonyms': ', '.join(syns[:6])})
pd.DataFrame(rows).sort_values('in_32_texts', ascending=False).reset_index(drop=True)

,term,in_32_texts,n_wordnet_senses,sample_synonyms
0,share,True,10,"portion, part, percentage, parcel, contributio..."
1,margin,True,6,"border, perimeter, security deposit, gross pro..."
2,stock,True,27,"inventory, gunstock, stock certificate, store,..."
3,interest,False,10,"involvement, sake, interestingness, stake, int..."
4,shares,False,10,"share, portion, part, percentage, parcel, cont..."
5,bond,False,14,"chemical bond, bond certificate, alliance, bai..."
6,security,False,9,"protection, certificate, surety, security depa..."
7,return,False,29,"tax return, income tax return, homecoming, com..."
8,principal,False,7,"school principal, head teacher, head, star, le..."
9,capital,False,11,"working capital, capital letter, uppercase, up..."


Last run: 2026-06-16 19:02:02


### Obs 2: Numbers carry sentiment, and direction reinforces it

In general, the sentiment in numbers is captured by the movement, and this movement is described using a verb/adjective prior to the number.‍ Numbers without movement are usually dates, and dates carry no sentiment.‍

**Action:** The method must leave the numbers and their direction intact.‍ The word describing the movement can be replaced as long as it follows the movement.‍

In [16]:
# watermark: AGLLM (AI-assisted content disclosure)
NUM = re.compile(r'\d[\d.,]*\s*%?')
df['n_numbers'] = df['text'].apply(lambda t: len(NUM.findall(t)))
df['has_number'] = df['n_numbers'] > 0
df['dir_up'] = df['text'].str.contains(
    r'\b(?:increase|increased|rose|grew|growth|up|higher|gain|gained)\b', case=False, regex=True) \
& df['has_number']
df['dir_down'] = df['text'].str.contains(
    r'\b(?:decrease|decreased|fell|fall|loss|lower|decline|declined|down|drop|dropped)\b', case=False, regex=True) \
& df['has_number']
df['dir_neutral'] = ~(df['dir_up'] | df['dir_down']) & df['has_number']
df.groupby('label_name').agg(
    texts=('text', 'size'),
    with_numbers=('has_number', 'sum'),
    avg_numbers=('n_numbers', 'mean'),
    up_words=('dir_up', 'sum'),
    down_words=('dir_down', 'sum'),
    neutral=('dir_neutral', 'sum')).round(2).reindex(du.LABEL_NAMES)

,texts,with_numbers,avg_numbers,up_words,down_words,neutral
label_name,,,,,,
negative,11,10,2.27,0,8,2
neutral,10,5,1.30,0,0,5
positive,11,9,3.55,8,1,1


Last run: 2026-06-16 19:02:02


In [17]:
# watermark: AGLLM (AI-assisted content disclosure)
#positive sentiment with number, and no captured "up word"
#understandable becaue number was actually a date, so no direction, sentiment came from other words
df[df['has_number'] & (df['label_name']=="positive") & (df['dir_up']==False)].text.tolist()

['Finnish energy company Fortum Oyj said on November 13 , 2007 it was granted an environmental permit to build a biofuel-fired combined heat and power CHP plant in Vartan harbor in eastern Stockholm .']

Last run: 2026-06-16 19:02:03


In [18]:
# watermark: AGLLM (AI-assisted content disclosure)
#negative sentiment with number, and no captured "down word"
#another example of date captured, and for the other there was really no direction
df[df['has_number'] & (df['label_name']=="negative") & (df['dir_down']==False)].text.tolist()

['Scanfil issued a profit warning on 10 April 2006 .',
 'Reported operating margin was a negative 5.9 % .']

Last run: 2026-06-16 19:02:03


### Obs 3: Generic business jargon containing neutral or polarized sentiment signal

There is business jargon that has no sentiment signal, or polarized sentiment signal (as also seen in EDA from part 1).‍ We categorize these terms below: 

1.‍ currencies
2.‍ time periods
3.‍ units (millions, thousands)
4.‍ organizational entities (e.g.‍ company, subsidiary, officials)
5.‍ financial metrics (e.g.‍ sales, profit, revenues)

These words are distributed across sentiments, even financial metrics with an inherent sentiment.‍ For example, profit is inherently positive, but what carries the sentiment is the words around it.‍ It may change in sentiment if used with a negative, e.g.‍ "decreased profit".‍

**Action:** The model can vary these words, even financial metrics that are inherently positive/negative, as long as the direction or opposite sentiment is retained.‍

In [19]:
# watermark: AGLLM (AI-assisted content disclosure)
generic_terms = ['quarter', 'year', 'period', 'company', 'eur', 'mn', 'million', 'market', 'subsidiary','officials','sales','profit','margin']

Last run: 2026-06-16 19:02:05


In [20]:
# watermark: AGLLM (AI-assisted content disclosure)
def class_counts(term):
    m = df['text'].str.contains(rf'\b{term}\b', case=False, regex=True)
    return df.loc[m, 'label_name'].value_counts().reindex(du.LABEL_NAMES).fillna(0).astype(int)

generic_tbl = pd.DataFrame({t: class_counts(t) for t in generic_terms}).T
generic_tbl['total'] = generic_tbl.sum(axis=1)
generic_tbl.sort_values('total', ascending=False)

label_name,negative,neutral,positive,total
profit,4,0,5,9
sales,2,2,4,8
eur,3,0,4,7
company,2,2,2,6
mn,2,0,3,5
year,1,0,2,3
period,1,0,2,3
million,1,1,1,3
market,1,2,0,3
subsidiary,0,1,0,1


Last run: 2026-06-16 19:02:06


In [21]:
# Example of financial term that is inherently positive (profit) but appears in negative sentiment text

# watermark: AGLLM (AI-assisted content disclosure)
neg_profit = df[df['text'].str.contains(r'\bprofit\b', case=False, regex=True) & (df['label_name'] == "negative")]['text']
neg_profit.iloc[0] if not neg_profit.empty else None

'Operating loss totaled EUR 0.8 mn , compared to a profit of EUR 0.5 mn .'

Last run: 2026-06-16 19:02:07


## Automatic Augmentation

Our goal is to create an automated augmentation model (not LLM-based) that produces new text with similar sentiment but with lexical diversity and structural variety.‍

What we found in the EDA is that we cannot simply change financial words according to synonyms, that the numbers describing the direction should be kept intact (but we can replace the words describing the movement), and that the area where we have the most freedom is generic financial jargon.‍

We thought of doing a rules-based word replacement; however, enumerating all possibilities is tedious, and this approach does not add structural variety.‍ What we want is a model that can paraphrase the text, and the method that does this is back-translation.‍

Back-translation works by taking a sentence, translating it into another language, and then translating it back into the original language.‍ This is the kind of thing you can do with Google Translate.‍ Because the translation almost never comes back word for word, what we get is a paraphrase of the original sentence, with the same meaning but different wording and structure.‍ This is not text generated by an LLM.‍ Back-translation uses a neural machine translation model, which is an encoder-decoder network built specifically to turn one language into another.‍ It reworks a sentence we already have rather than inventing new content from a prompt the way a generative model would.‍

To generate multiple paraphrased sentences of the same text, we translate through several different languages, one per identified family.‍ We choose a diversity of language families because the closer a language is to English, the more likely the translation will stay close to the original text.‍

We then measure the quality of augmentation based on the number of new tokens generated and how similar the word ordering is in comparison to the original.‍

### Back-translation

We use `MarianMTModel` from `transformers` to do the translation.‍ The pivot languages are:
1.‍ Spanish (es) - Indo-European, Romance, related to English
2.‍ German (de) - Indo-European, Germanic, close to English
3.‍ Russian (ru) - Indo-European, Slavic, related to English
4.‍ Finnish (fi) - Uralic
5.‍ Tagalog (tl) - Austronesian
6.‍ Arabic (ar) - Afro-Asiatic
7.‍ Chinese (zh) - Sino-Tibetan

We take note of the commit hashes of each translation model from the Hugging Face Hub for reproducibility.‍ The model setting that can significantly change the result is the decoding strategy, which we set to beam search to make it deterministic: it uses no random sampling and always returns the translation with the highest probability.‍ We chose a `beam width = 4` as a reasonable beam size (the number of candidate translations to choose from) that is not too greedy.‍ We set `max_length = 256`, which is longer than the length of the texts in the data.‍

With 32 texts and 7 translations, we expect a maximum of 32 x 7 = 224 candidate translations.‍ We omit the translations that did not produce a change from the original, based on a simple comparison of the text after normalizing whitespace and converting to lowercase.‍

To check whether we are achieving the goal of lexical diversity and structural variety, we check for new words produced in the translation, and the "edit distance", i.e.‍ the number of word-level changes needed to arrive at the original sentence.‍

In [21]:
# watermark: AGLLM (AI-assisted content disclosure)

PIVOTS = ['es', 'de', 'ru', 'fi', 'tl', 'ar', 'zh']
# Exact opus-mt commit hashes used to build the committed data/augmented_32.csv.
REVISIONS = {
    'Helsinki-NLP/opus-mt-en-es': '5bc4493d463cf000c1f0b50f8d56886a392ed4ab',
    'Helsinki-NLP/opus-mt-es-en': 'c96e2c5399ebfae4fc43d9669556b9afa74bb69d',
    'Helsinki-NLP/opus-mt-en-de': '6183067f769a302e3861815543b9f312c71b0ca4',
    'Helsinki-NLP/opus-mt-de-en': '1a922f3b32a8e809e17a47d4b32142d8105924e5',
    'Helsinki-NLP/opus-mt-en-ru': 'bb09c99d180016eac6819df3dae68edb1690fdee',
    'Helsinki-NLP/opus-mt-ru-en': 'fbd6dc73284f95536648512cc21d57f19191961a',
    'Helsinki-NLP/opus-mt-en-fi': '2f68f12519005b50f9e1afe6fd57b13add55e156',
    'Helsinki-NLP/opus-mt-fi-en': 'c4d8fff0f8f00b637f0578666f35cdacef354778',
    'Helsinki-NLP/opus-mt-en-tl': 'e46e1761492cb6a6fb9515a72bb55ca654815ca5',
    'Helsinki-NLP/opus-mt-tl-en': '8d2dee91f8b3fa48dda8ec06ee2af8f24970dcbc',
    'Helsinki-NLP/opus-mt-en-ar': '03087980e8ce753d64b3248ed0a912444545b840',
    'Helsinki-NLP/opus-mt-ar-en': 'c5b2a50db78d6be98ae207223f8d4f63bc7a0ff1',
    'Helsinki-NLP/opus-mt-en-zh': '408d9bc410a388e1d9aef112a2daba955b945255',
    'Helsinki-NLP/opus-mt-zh-en': 'cf109095479db38d6df799875e34039d4938aaa6',
}
_MODELS = {}

@torch.no_grad()
def translate(texts, model_name, batch_size=16, num_beams=4):
    if model_name not in _MODELS:
        rev = REVISIONS.get(model_name)
        tok = MarianTokenizer.from_pretrained(model_name, revision=rev)
        model = MarianMTModel.from_pretrained(model_name, revision=rev).to(DEVICE).eval()
        _MODELS[model_name] = (tok, model)
    tok, model = _MODELS[model_name]
    out = []
    for i in range(0, len(texts), batch_size):
        enc = tok(texts[i:i + batch_size], return_tensors='pt',
                  padding=True, truncation=True, max_length=256).to(DEVICE)
        gen = model.generate(**enc, num_beams=num_beams, max_length=256)  
        out.extend(tok.batch_decode(gen, skip_special_tokens=True))
    return out

def back_translate(texts, pivot):
    return translate(translate(texts, f'Helsinki-NLP/opus-mt-en-{pivot}'),
                     f'Helsinki-NLP/opus-mt-{pivot}-en')  # en -> pivot -> en

# Provenance for reproducibility: record what produced the committed CSV.
print(f'device={DEVICE} | torch={torch.__version__} | transformers={transformers.__version__}')
print('pivots:', PIVOTS, '| 14 model revisions pinned')

device=mps | torch=2.12.0 | transformers=5.10.1
pivots: ['es', 'de', 'ru', 'fi', 'tl', 'ar', 'zh'] | 14 model revisions pinned
Last run: 2026-06-15 20:33:43


In [22]:
# watermark: AGLLM (AI-assisted content disclosure)
# Helper functions used to filter and normalise the back-translation candidates.
_DIGITS = re.compile(r'\d+')

# digit_multiset: the sorted bag of digit groups in a text. Used to verify numbers are
# unchanged; ignores thousands separators / decimal style (e.g. "1,050.7" -> ['1','050','7']).
def digit_multiset(t): return sorted(_DIGITS.findall(str(t)))

# norm: lowercase + collapse whitespace. A canonical form for comparing/deduping sentences,
# so trivial case or spacing differences are not treated as different text.
def norm(t): return ' '.join(str(t).lower().split())

# numbers_preserved: True if a candidate keeps exactly the same digits as its source.
# This is the number-guard that protects Observation 2 (numbers carry the sentiment).
def numbers_preserved(src, cand): return digit_multiset(src) == digit_multiset(cand)

# clean: light, language-neutral normalisation of a translation output -- decode HTML
# entities (e.g. "&apos;" -> "'") and collapse any runs of extra whitespace.
def clean(t): return re.sub(r'\s{2,}', ' ', html.unescape(str(t))).strip()

Last run: 2026-06-16 19:02:17


### Run

In [23]:
# watermark: AGLLM (AI-assisted content disclosure)
src_texts = df['text'].tolist()
records = []
for pivot in PIVOTS:
    bt = back_translate(src_texts, pivot)
    for (_, row), cand in zip(df.iterrows(), bt):
        records.append({'source_id': row['id'], 'method': f'bt-{pivot}',
                        'label': row['label'], 'label_name': row['label_name'],
                        'src': row['text'], 'text': clean(cand)})
cand_df = pd.DataFrame(records)
cand_df['numbers_ok'] = [numbers_preserved(s, c) for s, c in zip(cand_df['src'], cand_df['text'])]
cand_df['is_new'] = cand_df['text'].map(norm) != cand_df['src'].map(norm)

orig_norm = set(map(norm, src_texts))
kept = (cand_df[cand_df['numbers_ok'] & cand_df['is_new']]
        .loc[lambda d: ~d['text'].map(norm).isin(orig_norm)]
        .drop_duplicates(subset='text'))

orig = df[['id', 'text', 'label', 'label_name']].assign(method='original', source_id=df['id'])
new = kept[['text', 'label', 'label_name', 'method', 'source_id']].copy()
new.insert(0, 'id', range(100000, 100000 + len(new)))
augmented = pd.concat([orig, new], ignore_index=True)[['id', 'text', 'label', 'label_name', 'method', 'source_id']]
augmented.to_csv('../data/augmented_32.csv', index=False)

print('kept per pivot:'); print(kept['method'].value_counts())
print(f"\nnumbers_ok: {int(cand_df['numbers_ok'].sum())}/{len(cand_df)}  |  total kept: {len(kept)}")
print(f"augmented_32.csv: {len(augmented)} rows = {len(orig)} originals + {len(new)} paraphrases")
print(augmented['label_name'].value_counts().reindex(du.LABEL_NAMES))

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

kept per pivot:
method
bt-es    32
bt-de    32
bt-tl    29
bt-ar    29
bt-ru    28
bt-fi    27
bt-zh    20
Name: count, dtype: int64

numbers_ok: 199/224  |  total kept: 197
augmented_32.csv: 229 rows = 32 originals + 197 paraphrases
label_name
negative    78
neutral     73
positive    78
Name: count, dtype: int64
Last run: 2026-06-15 20:35:46


### Review Samples

In general, we see a lot of restructuring of the sentences and some word replacement.‍ Languages that are near English (e.g.‍ es, de) have translations that stay close to the original, while the farther a language is from English (e.g.‍ Tagalog), the more word replacements and reordering we get.‍ The sentiments are retained.‍

Looking at the average new words per translation and the edit distance, the values are highest for languages that are far from the English family, confirming the earlier observations.‍

In [23]:
# watermark: AGLLM (AI-assisted content disclosure)
# Original vs its paraphrases — one source example per class.
aug = pd.read_csv('../data/augmented_32.csv')
src_text = labeled_32.set_index('id')['text']
example_ids = aug[aug.method == 'original'].groupby('label_name', group_keys=False).head(5)['source_id']
for sid in example_ids:
    grp = aug[aug.source_id == sid]
    print(f"\n=== source {sid} [{grp.iloc[0]['label_name']}] ===")
    print("ORIG:", src_text.get(sid))
    for _, r in grp[grp.method != 'original'].iterrows():
        print(f"[{r['method']}] {r['text']}")


=== source 6 [positive] ===
ORIG: Clothing retail chain Sepp+ñl+ñ 's sales increased by 8 % to EUR 155.2 mn , and operating profit rose to EUR 31.1 mn from EUR 17.1 mn in 2004 .
[bt-es] Sales of the Seppl clothing retail chain increased by 8 % to € 155.2 million , and operating profits rose to € 31.1 million , compared with € 17.1 million in 2004 .
[bt-de] Sales of the clothing retail chain Sepp+ñl+ñ increased by 8 % to EUR 155.2 million and operating profit increased from EUR 17.1 million in 2004 to EUR 31.1 million .
[bt-ru] Seppp+ni+Ni sales increased by 8% to Euro155.2 million, while operating gains increased to Euro31.1 million, compared to Euro17.1 million in 2004.
[bt-fi] Sales in the clothing chain increased by 8 per cent to EUR 155.2 million and operating profit to EUR 31.1 million from EUR 17.1 million in 2004 .
[bt-tl] The retail chain Sepp+ñol+ñ's sales rose by 8 %s in EUR 155.2 mn , and the cost of operating rose to the EUR 31.1 mn from EUR 17.1 mn in 2004 .
[bt-ar] Sales

In [24]:
# watermark: AGLLM (AI-assisted content disclosure)
# Diversity per translation (pivot): avg NEW WORDS produced (lexical) and avg normalized
# word-level EDIT DISTANCE vs the source (structural). 0 edit_dist = identical, 1 = fully different.
aug = pd.read_csv('../data/augmented_32.csv')
src_text = pd.read_csv('../data/labeled_32.csv').set_index('id')['text']
def _toks(t): return re.findall(r"[a-z]+|\d+", str(t).lower())            # ordered word/number tokens
def n_new_words(src, cand): return len(set(_toks(cand)) - set(_toks(src)))# word types not in the source
def norm_edit(src, cand):
    a, b = _toks(src), _toks(cand)
    return nltk.edit_distance(a, b) / max(len(a), len(b), 1)

para = aug[aug.method != 'original'].copy()
para['src'] = para['source_id'].map(src_text)
para['new_words'] = [n_new_words(s, t) for s, t in zip(para['src'], para['text'])]
para['edit_dist'] = [norm_edit(s, t) for s, t in zip(para['src'], para['text'])]
print('per translation (higher new_words = more lexical diversity; higher edit_dist = more restructuring):')
print(para.groupby('method').agg(n=('text','size'),
      avg_new_words=('new_words','mean'), avg_norm_edit_dist=('edit_dist','mean')).round(2)
      .sort_values('avg_norm_edit_dist', ascending=False))
print(f"\noverall: avg_new_words={para['new_words'].mean():.2f}  avg_norm_edit_dist={para['edit_dist'].mean():.3f}")


per translation (higher new_words = more lexical diversity; higher edit_dist = more restructuring):
         n  avg_new_words  avg_norm_edit_dist
method                                       
bt-zh   20           3.65                0.53
bt-ru   28           4.57                0.52
bt-ar   29           5.31                0.47
bt-fi   27           4.00                0.40
bt-de   32           3.00                0.37
bt-tl   29           3.86                0.33
bt-es   32           2.66                0.28

overall: avg_new_words=3.84  avg_norm_edit_dist=0.406
Last run: 2026-06-16 19:02:31


## Part 2d.‍ Data Generation WITH an LLM (1)

Now, the automated augmentation can also be done with an LLM.‍ The observations from earlier are inherent in data generation by an LLM:

1.‍ change financial words based on context
2.‍ invent the numbers following the direction
3.‍ change the neutral generic financial jargon

So, we only need some algorithmic choices to guide our generation:
1.‍ We make sure to only generate single-sentence financial news so that it is consistent in structure.‍ 
2.‍ We also take advantage of the fact that we know that the annotators used investor's point of view in determining the sentimenet.‍ So we will ask it to the sentence according to this sentiment.‍
3.‍ We can make the sentiments the same in value, stronger, or weaker, just so we have variety in the strength of the sentiment.‍
4.‍ When a specific company is used, we can also ask it to create fictional but realistic entities so that we do not create dependence on the same entity.‍
5.‍  We will only use the 32 labeled texts as "style anchors" so as not to overfit to the 32.‍ What this means is that we should make multiple calls to the prompt, generating data from a sample of the 32 each time.‍

### LLM generation parameters

For the model, we use `Groq` because it is fast and its free tier has higher rate limits compared to other LLM providers.‍ An important modeling choice is `TEMPERATURE`, which we set at 0.95 to create high lexical variety.‍ Following the API documentation advice, we only modify either `TEMPERATURE` or `top_p` so we did not touch `top_p`.‍ The default value for `top_p` is 1 which means we consider all tokens (even small probabilities).‍ In addition, it is also the setting most consistent with getting high lexical variety.‍  We choose 3 random same-class examples (e.g.‍ 3 of the 11 positives) and request 10 new sentences per call.‍ We loop this per sentiment and run repeatedly until we get 120 new examples per class (or until we reach a maximum of 30 calls).‍

In [25]:
# watermark: AGLLM (AI-assisted content disclosure)
# Part 2d config + helpers. Few-shot-prompted generation with the Groq API (free tier, open models).
# Get a free key at https://console.groq.com -> API Keys, then set GROQ_API_KEY.
GEN_MODEL           = "llama-3.3-70b-versatile"  # Groq open model; verify the current id at console.groq.com
TEMPERATURE         = 0.95                  # high temp -> lexical variety
N_PER_CLASS         = 120                   # balanced target per class
FEWSHOT_K           = 3                     # rotating same-class anchors per call
BATCH_SIZE          = 10                    # examples requested per API call
MAX_CALLS_PER_CLASS = 30                    # cost / rate-limit guard, per class

SYSTEM_PROMPT = (
    "You generate synthetic examples that mimic the Financial PhraseBank dataset: single-sentence "
    "snippets from English financial news about listed companies, labelled for sentiment from an "
    "INVESTOR's perspective - the likely effect on the company's value or share price, NOT the writer's tone.\n\n"
    "Label definitions:\n"
    "- positive: an investor reads it as good news (profit/sales/orders rising, beating estimates, "
    "margin expansion, contract wins with financial upside).\n"
    "- negative: bad news (losses, falling sales/profit, profit warnings, layoffs, downgrades, lost contracts).\n"
    "- neutral: factual with no clear value implication (management/HQ/name changes, listings, product or "
    "contract descriptions without an outcome, neutral guidance).\n\n"
    "Style:\n"
    "- One sentence, ~8-40 words, terse and factual like a news wire.\n"
    "- Fictional but realistic company names; vary the companies, regions and metrics.\n"
    "- Vary sentence structure and vocabulary across examples; never reuse a template.\n"
    "- Vary the STRENGTH of the sentiment across examples (some strong, some mild), but keep each one "
    "clearly within its label - a mild positive must still read as positive, never drift to neutral.\n"
    "- Do not make the label obvious from a single keyword; avoid cliches."
)

# Load the key: GROQ_API_KEY env var first, else the gitignored groq_api_key.txt at the repo root.
_keyfile = Path("..") / "groq_api_key.txt"
GROQ_API_KEY = os.environ.get("GROQ_API_KEY") or (_keyfile.read_text().strip() if _keyfile.exists() else None)
assert GROQ_API_KEY and "REPLACE" not in GROQ_API_KEY, \
    "Add your Groq key to groq_api_key.txt (repo root) or export GROQ_API_KEY."
client = Groq(api_key=GROQ_API_KEY)
rng = random.Random(SEED)         # reproducible anchor choices (SEED from the setup cell)
norm = lambda t: ' '.join(str(t).lower().split())

def build_user_prompt(label_name, n):
    pool = labeled_32.loc[labeled_32.label_name == label_name, 'text'].tolist()
    anchors = rng.sample(pool, min(FEWSHOT_K, len(pool)))            # rotating, same-class style anchors
    shown = "\n".join(f"- {a}" for a in anchors)
    return (f"Examples of *{label_name}* sentences - match their style and labelling, but write about "
            f"DIFFERENT companies and figures; do not paraphrase or reuse them:\n{shown}\n\n"
            f"Generate {n} {label_name} sentences. "
            f'Return ONLY a JSON array: [{{"text": "...", "label": "{label_name}"}}].')

def parse_examples(raw):
    m = re.search(r'\[.*\]', raw, re.S)                              # grab JSON array, ignore prose/fences
    if not m:
        return []
    try:
        items = json.loads(m.group(0))
    except json.JSONDecodeError:
        return []
    return [str(it['text']).strip() for it in items
            if isinstance(it, dict) and str(it.get('text', '')).strip()]

def generate_batch(label_name, n=BATCH_SIZE, retries=3):
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=GEN_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": build_user_prompt(label_name, n)},
                ],
                temperature=TEMPERATURE,
                max_tokens=2048,
            )
            return parse_examples(resp.choices[0].message.content or "")
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(5 * (attempt + 1))   # back off (free-tier rate limits / transient errors)

print(f"configured: model={GEN_MODEL} temp={TEMPERATURE} target={N_PER_CLASS}/class "
      f"| key loaded: {bool(GROQ_API_KEY)}")

configured: model=llama-3.3-70b-versatile temp=0.95 target=120/class | key loaded: True
Last run: 2026-06-16 19:07:26


### Run

In [27]:
# watermark: AGLLM (AI-assisted content disclosure)
# Generate a balanced set (N_PER_CLASS per class). 
seen = set(labeled_32['text'].map(norm))  
rows = []
for label_name in du.LABEL_NAMES:
    kept = 0
    for call_i in range(MAX_CALLS_PER_CLASS):
        if kept >= N_PER_CLASS:
            break
        for txt in generate_batch(label_name):
            k = norm(txt)
            if not k or k in seen:
                continue
            seen.add(k)
            rows.append({'text': txt, 'label': du.NAME2ID[label_name], 'label_name': label_name,
                         'method': f'llm:{GEN_MODEL}', 'source_id': pd.NA})
            kept += 1
            if kept >= N_PER_CLASS:
                break
    print(f"{label_name:9s}: {kept} kept")

gen = pd.DataFrame(rows)
gen.insert(0, 'id', range(200000, 200000 + len(gen)))   # 200000+ namespace (augmented used 100000+)
gen = gen[['id', 'text', 'label', 'label_name', 'method', 'source_id']]
gen.to_csv('../data/llm_generated.csv', index=False)

negative : 120 kept
neutral  : 120 kept
positive : 120 kept
Last run: 2026-06-15 20:37:52


In [28]:
# watermark: AGLLM (AI-assisted content disclosure)
print(f"\nllm_generated.csv: {len(gen)} rows")
print(gen['label_name'].value_counts().reindex(du.LABEL_NAMES))


llm_generated.csv: 360 rows
label_name
negative    120
neutral     120
positive    120
Name: count, dtype: int64
Last run: 2026-06-15 20:37:52


### Review the generated text

The generated texts seem to be different from the examples, and seem realistic.‍ 

In [29]:
# watermark: AGLLM (AI-assisted content disclosure)
# Sanity view: samples per class + lexical novelty vs the 32 (more new word types = more variety).
gen = pd.read_csv('../data/llm_generated.csv')
for ln in du.LABEL_NAMES:
    print(f"\n=== {ln} ===")
    for t in gen.loc[gen.label_name == ln, 'text'].head(3):
        print(" -", t)


=== negative ===
 - Nordic Telecom's quarterly earnings plummeted 12% year-over-year.
 - PetroLink's oil production declined 5.5% in the first half of 2022.
 - On 15 March, Baltic Bank announced a significant downgrade in its forecast.

=== neutral ===
 - The company's annual general meeting will be held on 15 May.
 - Further information is available on the investor relations section of the website.
 - The board of directors has appointed a new corporate secretary.

=== positive ===
 - Energia Verde's quarterly revenues jumped 12% to $23.5 million, driven by strong demand for solar panels.
 - Kaiser Steel reported a significant increase in shipments, with sales rising 15% year-over-year to $420 million.
 - Profit at Nordic Paper rose to $14.2 million from $8.5 million a year ago, beating analyst expectations.
Last run: 2026-06-15 20:37:52


## Comparison Augmented - Back Translation vs LLM vs OG

To compare the lexical diversity and structural variety of both methods, we calculate the number of new vocabulary words (for lexical diversity) and the number of unique n-grams (for structural variety).‍ The n-grams are an alternative because we cannot compare the LLM-generated text against a specific source sentence.‍

We can see from the table that the LLM generated a lot more new vocabulary than back-translation (BT): ~1000 more words than the original, versus back-translation, which generated only ~200 more.‍ This is expected because back-translation is not generative.‍ Distinct-3, which is the number of unique trigrams divided by total trigrams, describes the structural variety of the sentences.‍ Note that the metric rises with fewer sentences to compare.‍ However, the LLM has a higher value than BT (0.83 vs 0.57) even though the LLM has a larger corpus than BT.‍ This means BT contains a lot of repetitions because it is limited to what it was fed.‍

Thus the LLM generates more lexical diversity and structural variety.‍

In [30]:
# watermark: AGLLM (AI-assisted content disclosure)
# Diversity: the 32 baseline vs each method's OUTPUT (back-translation paraphrases, LLM-generated).
# NOTE: distinct-n shrink with corpus size (distinct-1 is the type-token ratio), so different-sized sets aren't
# directly comparable on those — the fair cross-set comparisons are vocab size and new word types.
_toks = lambda t: re.findall(r"[a-z]+|\d+", str(t).lower())
def _ngrams(tk, n): return [tuple(tk[i:i+n]) for i in range(len(tk) - n + 1)]
def diversity(texts):
    tp = [_toks(t) for t in texts]; allt = [w for tk in tp for w in tk]; vocab = set(allt)
    dn = lambda n: (lambda g: len(set(g))/len(g) if g else 0.0)([ng for tk in tp for ng in _ngrams(tk, n)])
    return {'sentences': len(texts), 'total tokens': len(allt), 'vocab (types)': len(vocab),
            'distinct-1': round(dn(1), 3), 'distinct-2': round(dn(2), 3), 'distinct-3': round(dn(3), 3)}

aug = pd.read_csv('../data/augmented_32.csv')
aug_para = aug.loc[aug.method != 'original', 'text']   # back-translation output only (exclude the 32)
sets = {'the 32': labeled_32['text'], 'back-translation': aug_para, 'LLM-generated': gen['text']}
stats = {name: diversity(texts) for name, texts in sets.items()}
cols = list(sets)
print(f"{'metric':22s}" + "".join(f"{c:>17s}" for c in cols))
for k in stats[cols[0]]:
    print(f"{k:22s}" + "".join(f"{str(stats[c][k]):>17s}" for c in cols))

bv = set().union(*[set(_toks(t)) for t in labeled_32['text']])
print("\nnew word types vs the 32:")
for name in ['back-translation', 'LLM-generated']:
    v = set().union(*[set(_toks(t)) for t in sets[name]])
    print(f"  {name:18s}{len(v - bv)}")

metric                           the 32 back-translation    LLM-generated
sentences                            32              197              360
total tokens                        712             4200             5287
vocab (types)                       341              552             1218
distinct-1                        0.479            0.131             0.23
distinct-2                        0.871            0.387            0.607
distinct-3                        0.977            0.574            0.811

new word types vs the 32:
  back-translation  220
  LLM-generated     1042
Last run: 2026-06-15 20:37:52
